# Q2 — Decision Tree Classifier

A scikit-learn `DecisionTreeClassifier` with tree visualization, applied to the
BSDS500-derived edge-pixel dataset.

## Setup

This notebook expects to be run from the project root, alongside a `common.py`
(or with the helper cell below), and with the following layout already in place:

```
project_root/
├── archive/                # BSDS500 images + ground_truth
├── data/bsds_features.csv  # produced by the feature-extraction notebook
├── results/figures/
└── results/metrics/
```

If you don't have a `common.py` file in this directory, run the cell below first —
it defines the same `load_split` / `save_metrics` helpers used across all seven
questions so this notebook is self-contained.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().resolve()
DATA_CSV = ROOT / "data" / "bsds_features.csv"
FIG_DIR = ROOT / "results" / "figures"
METRIC_DIR = ROOT / "results" / "metrics"
FIG_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = ["R", "G", "B", "gray", "grad_mag", "grad_dir",
                "laplacian", "local_std", "x_norm", "y_norm"]
LABEL_COL = "is_edge"
RANDOM_STATE = 42


def load_split(test_size=0.2, scale=True):
    df = pd.read_csv(DATA_CSV)
    X = df[FEATURE_COLS].values.astype(np.float64)
    y = df[LABEL_COL].values.astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=RANDOM_STATE, stratify=y
    )

    if scale:
        mu, sigma = X_train.mean(axis=0), X_train.std(axis=0)
        sigma[sigma == 0] = 1.0
        X_train = (X_train - mu) / sigma
        X_test = (X_test - mu) / sigma

    return X_train, X_test, y_train, y_test


def save_metrics(name, d):
    path = METRIC_DIR / f"{name}.json"
    with open(path, "w") as f:
        json.dump(d, f, indent=2, default=float)
    print(f"saved metrics -> {path}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Load data
Decision trees don't need feature scaling; keep raw feature values.

In [ ]:
X_train, X_test, y_train, y_test = load_split(scale=False)
print(f"train={X_train.shape}, test={X_test.shape}")

## Fit a depth-6 tree

In [ ]:
clf = DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, random_state=42)
clf.fit(X_train, y_train)

preds = clf.predict(X_test)
acc = accuracy_score(y_test, preds)
cm = confusion_matrix(y_test, preds)
report = classification_report(y_test, preds, target_names=["non-edge", "edge"])
print(f"Test accuracy: {acc:.4f}")
print(report)

## Feature importances

In [ ]:
importances = sorted(zip(FEATURE_COLS, clf.feature_importances_),
                      key=lambda t: -t[1])
print("Feature importances:")
for name, imp in importances:
    print(f"  {name:12s} {imp:.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
names, vals = zip(*importances)
ax.barh(names, vals, color="#4C72B0")
ax.invert_yaxis()
ax.set_xlabel("importance")
ax.set_title("Decision Tree feature importances")
plt.tight_layout()
plt.savefig(FIG_DIR / "q2_tree_feature_importance.png", bbox_inches="tight")
plt.show()

## Full tree visualization

In [ ]:
fig, ax = plt.subplots(figsize=(26, 12))
plot_tree(clf, feature_names=FEATURE_COLS, class_names=["non-edge", "edge"],
          filled=True, rounded=True, fontsize=7, ax=ax)
ax.set_title(f"Decision Tree (max_depth=6)  test accuracy={acc:.3f}")
plt.savefig(FIG_DIR / "q2_tree_full.png", bbox_inches="tight", dpi=150)
plt.show()

## A shallower tree for a readable diagram

In [ ]:
clf_shallow = DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=42)
clf_shallow.fit(X_train, y_train)
preds_shallow = clf_shallow.predict(X_test)
acc_shallow = accuracy_score(y_test, preds_shallow)

fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(clf_shallow, feature_names=FEATURE_COLS, class_names=["non-edge", "edge"],
          filled=True, rounded=True, fontsize=10, ax=ax)
ax.set_title(f"Decision Tree (max_depth=3, readable)  test accuracy={acc_shallow:.3f}")
plt.savefig(FIG_DIR / "q2_tree_shallow.png", bbox_inches="tight", dpi=150)
plt.show()

## Parameter modification: sensitivity to `max_depth`

In [ ]:
depth_values = [1, 2, 3, 4, 5, 6, 8, 10, 15, None]
depth_accs = []
for d in depth_values:
    c = DecisionTreeClassifier(max_depth=d, min_samples_leaf=20, random_state=42)
    c.fit(X_train, y_train)
    a = accuracy_score(y_test, c.predict(X_test))
    depth_accs.append(a)
    print(f"max_depth={str(d):5s} accuracy={a:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
x_labels = [str(d) if d is not None else "None" for d in depth_values]
ax.plot(x_labels, depth_accs, marker="o")
ax.set_xlabel("max_depth")
ax.set_ylabel("test accuracy")
ax.set_title("Decision Tree: accuracy vs max_depth")
plt.tight_layout()
plt.savefig(FIG_DIR / "q2_depth_sensitivity.png", bbox_inches="tight")
plt.show()

## Save metrics

In [ ]:
save_metrics("q2_decision_tree", {
    "depth6_test_accuracy": acc,
    "depth3_test_accuracy": acc_shallow,
    "confusion_matrix": cm.tolist(),
    "classification_report": report,
    "feature_importances": {n: float(v) for n, v in importances},
    "depth_sweep": {"depths": x_labels, "accuracies": depth_accs},
})